# الفصل الخامس والعشرون: الاختبار (Testing) باستخدام pytest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-25.ipynb)

> **الوقت المتوقع:** ~10 دقيقة قراءة · **16** قسمًا · **25** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **25.1** لماذا نختبر الكود؟
- **25.2** المفاهيم الأساسية للاختبار
- **25.3** البدء مع pytest
- **25.4** `assert`: قلب كل اختبار
- **25.5** Fixtures: تجهيز بيانات الاختبار بكفاءة
- **25.6** Parametrize: اختبار حالات متعددة بكفاءة
- **25.7** اختبار الأصناف (Classes) والكود الكائني
- **25.8** قياس تغطية الاختبارات (Coverage)
- **25.9** اختبار الكود غير المتزامن (async)
- **25.10** Mocking: محاكاة الأجزاء الخارجية
- **25.11** مشروع تطبيقي متكامل: اختبار نظام إدارة مخزون
- **25.12** متى تكتب الاختبارات؟ (TDD بإيجاز)
- **25.13** تمارين
- **25.14** اختبر فهمك (Quiz)
- **25.15** ملخص الفصل (Cheat Sheet)
- **25.16** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["pytest"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q pytest

> ℹ️ نكتب ملفات الاختبار بـ `%%writefile` ثم نشغّل `!pytest`.

## 25.1 لماذا نختبر الكود؟

تخيّل هذا السيناريو: لديك دالة `calculate_discount()` تعمل بشكل ممتاز منذ أشهر. يأتي زميلك ويُعدّل دالة أخرى في نفس الملف لإضافة ميزة جديدة، دون أن يعلم أن تعديله الصغير كسر منطق الخصم بطريقة غير متوقعة. لا أحد يلاحظ المشكلة حتى يشتكي عميل حقيقي من حساب خاطئ لفاتورته — بعد أسابيع من نشر الكود!

هذا بالضبط ما تحلّه **الاختبارات الآلية (Automated Tests)**: برامج صغيرة تتحقق تلقائيًا من أن كودك يعمل كما هو متوقع، وتُشغَّل في ثوانٍ معدودة، وتُخبرك **فورًا** إذا كسر أي تعديل جديد شيئًا كان يعمل سابقًا.

> **🔄 مقارنة: الاختبار اليدوي مقابل الاختبار الآلي**
>
> | المعيار | الاختبار اليدوي | الاختبار الآلي (pytest) |
> |---|---|---|
> | السرعة | دقائق إلى ساعات | ثوانٍ |
> | التكرار | يُنسى غالبًا، أو يُهمَل لضيق الوقت | يُشغَّل تلقائيًا في كل مرة (حتى قبل كل Commit) |
> | التغطية | جزئية، تعتمد على مزاج المختبِر | شاملة ومتسقة |
> | الثقة عند التعديل | قلق دائم من «هل كسرتُ شيئًا؟» | ثقة: «لو كسرتُ شيئًا، سأعرف فورًا» |
> | التوثيق الضمني | لا يوجد | الاختبارات نفسها توثّق كيف يُفترض أن يعمل الكود |

في هذا الفصل، سنتعلّم **`pytest`** — المكتبة المعيارية والأكثر استخدامًا للاختبار في عالم بايثون — من الأساسيات حتى بناء مجموعة اختبارات حقيقية لمشروع كامل.

## 25.2 المفاهيم الأساسية للاختبار

### أنواع الاختبارات (نظرة عامة)

| النوع | ماذا يختبر؟ | مثال |
|---|---|---|
| **اختبار الوحدة (Unit Test)** | دالة أو صنف **واحد منفرد**، بمعزل عن باقي النظام | هل `calculate_total(10, 2)` تُرجع `20`؟ |
| **اختبار التكامل (Integration Test)** | تفاعل عدة أجزاء من النظام معًا | هل يعمل حفظ البيانات في قاعدة البيانات الفعلية بشكل صحيح؟ |
| **اختبار النظام الشامل (End-to-End)** | النظام كاملاً من منظور المستخدم النهائي | هل يمكن للمستخدم تسجيل الدخول، إضافة منتج، وإتمام الشراء؟ |

في هذا الفصل سنركّز بشكل أساسي على **اختبارات الوحدة**، لأنها الأساس الذي يُبنى عليه كل شيء آخر، وهي الأسرع والأسهل للبدء بها.

### هيكل الاختبار النموذجي: Arrange-Act-Assert (AAA)

كل اختبار جيد يتبع عادة نمطًا من ثلاث خطوات:

1. **Arrange (التحضير)**: جهّز البيانات والظروف اللازمة للاختبار.
2. **Act (التنفيذ)**: نفّذ الكود الذي تريد اختباره.
3. **Assert (التحقق)**: تحقّق أن النتيجة مطابقة لما تتوقعه.

In [ ]:
def test_addition():
    # Arrange (التحضير)
    a = 5
    b = 3

    # Act (التنفيذ)
    result = a + b

    # Assert (التحقق)
    assert result == 8

## 25.3 البدء مع pytest

### التثبيت

```bash
uv add --dev pytest
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### أول اختبار

لنفترض لدينا ملف `calculator.py` بدالة بسيطة:

In [ ]:
# src/calculator.py
def add(a: float, b: float) -> float:
    """جمع رقمين."""
    return a + b

def divide(a: float, b: float) -> float:
    """قسمة رقمين، مع رفع استثناء عند القسمة على صفر."""
    if b == 0:
        raise ValueError("لا يمكن القسمة على صفر")
    return a / b

نكتب ملف الاختبار في `tests/test_calculator.py`:

```python
# tests/test_calculator.py
from calculator import add, divide

def test_add_two_positive_numbers():
    result = add(2, 3)
    assert result == 5

def test_add_negative_numbers():
    result = add(-2, -3)
    assert result == -5

def test_divide_normal_case():
    result = divide(10, 2)
    assert result == 5
```

🧾 **للقراءة فقط** — الوحدة `calculator` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

تشغيل الاختبارات:

```bash
uv run pytest
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
======================== test session starts =========================
collected 3 items

tests/test_calculator.py ...                                    [100%]

========================= 3 passed in 0.02s =========================
```

> **💡 نصيحة خبير: قواعد تسمية صارمة يعتمد عليها pytest**
> يكتشف pytest الاختبارات تلقائيًا بناءً على قواعد تسمية محددة — **احفظها جيدًا**:
> - ملفات الاختبار: يجب أن تبدأ بـ `test_` أو تنتهي بـ `_test.py` (مثل `test_calculator.py`)
> - دوال الاختبار: يجب أن تبدأ بـ `test_` (مثل `def test_add_numbers():`)
> - إذا نسيت هذه البادئة، سيتجاهل pytest اختبارك **بصمت** دون أي تحذير!

### تنظيم المشروع مع الاختبارات

```
my_project/
├── pyproject.toml
├── src/
│   └── calculator.py
└── tests/
    ├── __init__.py
    └── test_calculator.py
```

> **💡 نصيحة خبير:** ضَع ملفات الاختبار في مجلد `tests/` منفصل عن الكود المصدري في `src/`. هذا يُبقي المشروع منظّمًا، ويُسهّل استبعاد الاختبارات عند تحزيم (packaging) المشروع للنشر النهائي.

## 25.4 `assert`: قلب كل اختبار

كما تعلمنا في الفصل 11، `assert` تتحقق من شرط وترفع `AssertionError` إن لم يتحقق. pytest يستخدم `assert` العادية في بايثون، لكنه **يُحسّن رسائل الخطأ تلقائيًا** لتكون واضحة جدًا عند الفشل:

In [ ]:
def test_add_fails_example():
    result = add(2, 2)
    assert result == 5  # سيفشل عمدًا لنرى رسالة الخطأ

```bash
uv run pytest tests/test_calculator.py::test_add_fails_example -v
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
FAILED tests/test_calculator.py::test_add_fails_example - assert 4 == 5
    def test_add_fails_example():
        result = add(2, 2)
>       assert result == 5
E       assert 4 == 5
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

لاحظ كيف أن pytest أخبرنا بالضبط: «النتيجة الفعلية كانت 4، لكننا توقعنا 5» — دون أي كتابة إضافية من جانبنا!

### أنواع مختلفة من التحقق

In [ ]:
def test_various_assertions():
    assert 5 > 3                           # مقارنة
    assert "hello" in "hello world"        # الاحتواء
    assert [1, 2, 3] == [1, 2, 3]           # تساوي القوائم
    assert isinstance(5, int)               # النوع
    assert not False                        # النفي
    assert {"a": 1} == {"a": 1}             # تساوي القواميس

### اختبار الاستثناءات: `pytest.raises`

```python
import pytest
from calculator import divide

def test_divide_by_zero_raises_error():
    with pytest.raises(ValueError):
        divide(10, 0)

def test_divide_by_zero_error_message():
    with pytest.raises(ValueError, match="لا يمكن القسمة على صفر"):
        divide(10, 0)
```

🧾 **للقراءة فقط** — الوحدة `calculator` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

> **⚠️ تحذير: اختبار الحالة السعيدة فقط لا يكفي**
> من أشيع الأخطاء عند كتابة الاختبارات هو اختبار **فقط** الحالات التي «تعمل بشكل صحيح» (Happy Path)، وتجاهل الحالات الاستثنائية والحدّية (Edge Cases). اختبار جيد يغطي:
> - الحالة الطبيعية (`divide(10, 2)`)
> - الحالات الحدّية (`divide(0, 5)`, قيم سالبة)
> - حالات الخطأ المتوقعة (`divide(10, 0)`)

### اختبار القيم التقريبية (الأعداد العشرية)

تذكّر من الفصل 5 أن الأعداد العشرية قد تحمل أخطاء دقة صغيرة:

```python
def test_float_addition_naive():
    assert 0.1 + 0.2 == 0.3  # ❌ قد يفشل! بسبب دقة الفاصلة العائمة

def test_float_addition_correct():
    assert 0.1 + 0.2 == pytest.approx(0.3)  # ✅ يتحقق بتسامح معقول من الفروق الدقيقة جدًا
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

## 25.5 Fixtures: تجهيز بيانات الاختبار بكفاءة

غالبًا ما تحتاج عدة اختبارات إلى نفس «التحضير» (Arrange). بدلاً من تكرار نفس الكود في كل دالة اختبار، نستخدم **Fixtures** — دوال خاصة تُحضّر البيانات أو الموارد وتُمررها تلقائيًا للاختبارات التي تحتاجها.

In [ ]:
import pytest

@pytest.fixture
def sample_cart():
    """Fixture تُجهّز سلة تسوّق تجريبية لاستخدامها في عدة اختبارات."""
    return {
        "items": [
            {"name": "لابتوب", "price": 3500, "qty": 1},
            {"name": "ماوس", "price": 50, "qty": 2},
        ],
    }

def test_cart_has_two_item_types(sample_cart):
    assert len(sample_cart["items"]) == 2

def test_cart_total_quantity(sample_cart):
    total_qty = sum(item["qty"] for item in sample_cart["items"])
    assert total_qty == 3

لاحظ: كل دالة اختبار تحتاج `sample_cart` تستقبلها ببساطة **كمعامل بنفس الاسم**، وpytest يتكفّل تلقائيًا باستدعاء الـ fixture وتمرير نتيجتها.

> **💡 نصيحة خبير:** هذا يُشبه تمامًا مفهوم **Dependency Injection** الذي رأيناه في الفصل 24 (مبدأ DIP) — الاختبار «يطلب» ما يحتاجه، وpytest «يُحضره» له، دون أن يهتم الاختبار بكيفية إنشائه.

### نطاق الـ Fixtures (Scope)

In [ ]:
@pytest.fixture(scope="function")  # الافتراضي: يُعاد إنشاؤها لكل اختبار
def fresh_data():
    return []

@pytest.fixture(scope="module")  # تُنشأ مرة واحدة فقط لكل ملف اختبار
def database_connection():
    print("🔌 فتح اتصال بقاعدة البيانات...")
    connection = "وهمي-اتصال"
    yield connection  # القيمة التي تُمرَّر للاختبارات
    print("🔌 إغلاق الاتصال...")  # كود التنظيف، يُنفَّذ بعد انتهاء كل الاختبارات

> **جدول مرجعي: أنواع نطاق الـ Fixtures**
>
> | Scope | يُنشأ | الاستخدام المناسب |
> |---|---|---|
> | `function` (افتراضي) | قبل كل دالة اختبار | البيانات البسيطة الخفيفة |
> | `class` | مرة لكل صنف اختبار | بيانات مشتركة بين اختبارات مترابطة |
> | `module` | مرة لكل ملف اختبار | موارد أثقل (اتصال قاعدة بيانات وهمية) |
> | `session` | مرة واحدة لكل تشغيل pytest بالكامل | موارد باهظة جدًا (نادرة الاستخدام) |

### Fixtures مع `yield` للتنظيف (Setup/Teardown)

In [ ]:
import pytest
from pathlib import Path

@pytest.fixture
def temp_file(tmp_path):
    """إنشاء ملف مؤقت للاختبار، وحذفه تلقائيًا بعد الانتهاء."""
    file_path = tmp_path / "test_data.txt"
    file_path.write_text("بيانات تجريبية", encoding="utf-8")

    yield file_path  # الاختبار يستخدم هذه القيمة

    # هذا الجزء يُنفَّذ تلقائيًا بعد انتهاء الاختبار (تنظيف)
    print(f"تنظيف: {file_path}")

def test_file_content(temp_file):
    content = temp_file.read_text(encoding="utf-8")
    assert content == "بيانات تجريبية"

> **💡 نصيحة خبير:** `tmp_path` هو fixture **مدمج** في pytest نفسه! ينشئ مجلدًا مؤقتًا فريدًا لكل اختبار، ويحذفه تلقائيًا لاحقًا — مثالي لاختبار دوال تتعامل مع الملفات (كما تعلمنا في الفصل 16) دون ترك «فوضى» في نظام الملفات الحقيقي.

## 25.6 Parametrize: اختبار حالات متعددة بكفاءة

بدلاً من كتابة دالة اختبار منفصلة لكل حالة، نستخدم `@pytest.mark.parametrize` لتشغيل نفس الاختبار بمدخلات مختلفة:

```python
import pytest
from calculator import add

@pytest.mark.parametrize(
    "a, b, expected",
    [
        (2, 3, 5),
        (-1, 1, 0),
        (0, 0, 0),
        (-5, -5, -10),
        (100, 200, 300),
    ],
)
def test_add_multiple_cases(a, b, expected):
    assert add(a, b) == expected
```

🧾 **للقراءة فقط** — الوحدة `calculator` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

```bash
uv run pytest -v
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
tests/test_calculator.py::test_add_multiple_cases[2-3-5] PASSED
tests/test_calculator.py::test_add_multiple_cases[-1-1-0] PASSED
tests/test_calculator.py::test_add_multiple_cases[0-0-0] PASSED
tests/test_calculator.py::test_add_multiple_cases[-5--5--10] PASSED
tests/test_calculator.py::test_add_multiple_cases[100-200-300] PASSED
```

5 اختبارات من دالة واحدة فقط! هذا يُقلّل التكرار بشكل كبير (تطبيق مباشر لمبدأ DRY من الفصل 24).

> **💡 نصيحة خبير:** استخدم `parametrize` بسخاء، خصوصًا لاختبار **الحالات الحدّية** (قيم صفرية، سالبة، فارغة، قصوى) بكفاءة دون تكرار الكود.

## 25.7 اختبار الأصناف (Classes) والكود الكائني

In [ ]:
# src/bank_account.py
class InsufficientFundsError(Exception):
    """يُرفع عند محاولة سحب مبلغ أكبر من الرصيد المتاح."""
    pass

class BankAccount:
    def __init__(self, owner: str, balance: float = 0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("مبلغ الإيداع يجب أن يكون موجبًا")
        self.balance += amount

    def withdraw(self, amount: float) -> None:
        if amount > self.balance:
            raise InsufficientFundsError("الرصيد غير كافٍ")
        self.balance -= amount

```python
# tests/test_bank_account.py
import pytest
from bank_account import BankAccount, InsufficientFundsError

@pytest.fixture
def account():
    """حساب بنكي تجريبي برصيد ابتدائي 100."""
    return BankAccount(owner="أحمد", balance=100)

class TestDeposit:
    """تجميع كل اختبارات عملية الإيداع في صنف واحد."""

    def test_deposit_increases_balance(self, account):
        account.deposit(50)
        assert account.balance == 150

    def test_deposit_negative_amount_raises_error(self, account):
        with pytest.raises(ValueError):
            account.deposit(-10)

    def test_deposit_zero_raises_error(self, account):
        with pytest.raises(ValueError):
            account.deposit(0)

class TestWithdraw:
    """تجميع كل اختبارات عملية السحب في صنف واحد."""

    def test_withdraw_decreases_balance(self, account):
        account.withdraw(30)
        assert account.balance == 70

    def test_withdraw_more_than_balance_raises_error(self, account):
        with pytest.raises(InsufficientFundsError):
            account.withdraw(200)

    def test_withdraw_exact_balance_succeeds(self, account):
        account.withdraw(100)
        assert account.balance == 0
```

🧾 **للقراءة فقط** — الوحدة `bank_account` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

> **💡 نصيحة خبير:** تجميع الاختبارات المترابطة داخل أصناف (`class TestDeposit:`) يُنظّم مجموعة الاختبارات بوضوح، خصوصًا في المشاريع الكبيرة. لاحظ أن أسماء هذه الأصناف يجب أن تبدأ بـ `Test` (وليس `test_`) ليكتشفها pytest تلقائيًا.

## 25.8 قياس تغطية الاختبارات (Coverage)

**تغطية الاختبار (Test Coverage)** تقيس: **أي نسبة من كودك** تُنفَّذ فعليًا عند تشغيل الاختبارات؟

```bash
uv add --dev pytest-cov
uv run pytest --cov=src --cov-report=term-missing
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
---------- coverage: platform linux, python 3.14.0 -----------
Name                      Stmts   Miss  Cover   Missing
-------------------------------------------------------
src/calculator.py             8      0   100%
src/bank_account.py          15      2    87%   23-24
-------------------------------------------------------
TOTAL                        23      2    91%
```

هذا يُخبرنا بالضبط: الأسطر 23-24 في `bank_account.py` **لم تُختبر أبدًا**. يمكننا الآن إضافة اختبار يغطيها تحديدًا.

> **⚠️ تحذير: التغطية 100% ليست الهدف النهائي**
> تغطية 100% تعني فقط أن كل سطر **نُفِّذ** مرة واحدة على الأقل أثناء الاختبارات — وليس بالضرورة أن كل **سيناريو منطقي** اختبرناه بعمق. يمكن أن تحصل على تغطية 100% مع اختبارات سطحية جدًا لا تتحقق من الحالات الحدّية أو المنطق الدقيق. **الهدف الحقيقي** هو اختبارات ذات معنى، وليس مجرد رقم تغطية مرتفع. استخدم نسبة التغطية كـ **دليل** لاكتشاف الكود غير المُختبَر إطلاقًا، وليس كهدف نهائي بحد ذاته.

## 25.9 اختبار الكود غير المتزامن (async)

تذكّر الفصل 17؟ اختبار دوال `async` يتطلب إضافة بسيطة:

```bash
uv add --dev pytest-asyncio
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

In [ ]:
# src/async_utils.py
import asyncio

async def fetch_data() -> str:
    await asyncio.sleep(0.1)
    return "بيانات"

```python
# tests/test_async_utils.py
import pytest
from async_utils import fetch_data

@pytest.mark.asyncio
async def test_fetch_data():
    result = await fetch_data()
    assert result == "بيانات"
```

🧾 **للقراءة فقط** — الوحدة `async_utils` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

في `pyproject.toml`، يجب تفعيل الوضع التلقائي:

```toml
[tool.pytest.ini_options]
asyncio_mode = "auto"
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

## 25.10 Mocking: محاكاة الأجزاء الخارجية

ماذا لو أردنا اختبار دالة تعتمد على طلب شبكة حقيقي (كما تعلمنا في الفصل 21، Web Scraping)؟ لا نريد أن تعتمد اختباراتنا على اتصال إنترنت فعلي (بطيء، وغير مضمون النجاح دائمًا). الحل: **Mocking** — استبدال الجزء الخارجي بـ«نسخة وهمية» تتصرف كما نُريدها بالضبط.

In [ ]:
# src/weather.py
import requests

def get_temperature(city: str) -> float:
    """جلب درجة الحرارة الحالية لمدينة عبر API خارجي."""
    response = requests.get(f"https://api.weather.example.com/{city}", timeout=10)
    data = response.json()
    return data["temperature"]

```python
# tests/test_weather.py
from unittest.mock import MagicMock, patch

from weather import get_temperature

def test_get_temperature_with_mock():
    # استبدال requests.get بنسخة وهمية لا تتصل فعليًا بالإنترنت
    with patch("weather.requests.get") as mock_get:
        mock_response = MagicMock()
        mock_response.json.return_value = {"temperature": 25.5}
        mock_get.return_value = mock_response

        result = get_temperature("الرياض")

        assert result == 25.5
        mock_get.assert_called_once()  # التحقق من أن الدالة استُدعيت فعلاً
```

🧾 **للقراءة فقط** — الوحدة `weather` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

> **💡 نصيحة خبير:** استخدم Mocking لأي اعتماد خارجي بطيء، غير موثوق، أو مكلف: طلبات الشبكة، قواعد البيانات، إرسال بريد إلكتروني فعلي، أو حتى `datetime.now()` عند اختبار منطق يعتمد على الوقت الحالي. هذا يجعل اختباراتك **سريعة وموثوقة ومستقلة تمامًا** عن العالم الخارجي.

> **🔄 مقارنة: اختبار حقيقي مقابل اختبار بـ Mock**
>
> | | اختبار حقيقي (بدون Mock) | اختبار بـ Mock |
> |---|---|---|
> | السرعة | بطيء (ينتظر الشبكة فعليًا) | سريع جدًا (فوري) |
> | الموثوقية | يفشل إذا كان الإنترنت معطلاً أو الـ API متوقفًا | يعمل دائمًا بشكل متوقع |
> | الواقعية | يختبر التكامل الفعلي | يفترض سلوك الـ API، قد يُخفي مشاكل حقيقية |
> | الاستخدام المناسب | اختبارات تكامل قليلة ومتعمَّدة | معظم اختبارات الوحدة |

## 25.11 مشروع تطبيقي متكامل: اختبار نظام إدارة مخزون

سنكتب اختبارات شاملة لنظام إدارة مخزون بسيط، مغطّين كل ما تعلمناه: fixtures، parametrize، اختبار استثناءات، mocking، وتنظيم الاختبارات في أصناف.

### هيكلية المشروع

```
inventory_system/
├── pyproject.toml
├── src/
│   └── inventory/
│       ├── __init__.py
│       ├── models.py
│       └── inventory_manager.py
└── tests/
    ├── __init__.py
    ├── conftest.py
    └── test_inventory_manager.py
```

```bash
uv init inventory_system
cd inventory_system
uv add --dev pytest pytest-cov
mkdir -p src/inventory tests
touch src/inventory/__init__.py tests/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### `src/inventory/models.py`

```python
"""نماذج بيانات نظام إدارة المخزون."""

from dataclasses import dataclass

@dataclass
class Product:
    """يمثّل منتجًا في المخزون."""

    sku: str
    name: str
    price: float
    quantity: int

class ProductNotFoundError(Exception):
    """يُرفع عند البحث عن منتج غير موجود."""
    pass

class InsufficientStockError(Exception):
    """يُرفع عند محاولة بيع كمية أكبر من المتوفر."""
    pass

class DuplicateSKUError(Exception):
    """يُرفع عند محاولة إضافة منتج برمز SKU موجود مسبقًا."""
    pass
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/inventory/inventory_manager.py`

```python
"""منطق إدارة المخزون: إضافة، بيع، واستعلام عن المنتجات."""

from inventory.models import (
    DuplicateSKUError,
    InsufficientStockError,
    Product,
    ProductNotFoundError,
)

class InventoryManager:
    """يدير مجموعة من المنتجات داخل المخزون."""

    def __init__(self):
        self._products: dict[str, Product] = {}

    def add_product(self, product: Product) -> None:
        """إضافة منتج جديد إلى المخزون.

        Raises:
            DuplicateSKUError: إذا كان رمز SKU مستخدمًا مسبقًا.
        """
        if product.sku in self._products:
            raise DuplicateSKUError(f"المنتج {product.sku} موجود بالفعل")
        self._products[product.sku] = product

    def get_product(self, sku: str) -> Product:
        """الحصول على منتج بواسطة رمزه.

        Raises:
            ProductNotFoundError: إذا لم يوجد منتج بهذا الرمز.
        """
        if sku not in self._products:
            raise ProductNotFoundError(f"المنتج {sku} غير موجود")
        return self._products[sku]

    def sell(self, sku: str, quantity: int) -> float:
        """بيع كمية من منتج، وإرجاع الإيراد الناتج.

        Raises:
            ProductNotFoundError: إذا لم يوجد المنتج.
            ValueError: إذا كانت الكمية المطلوبة غير موجبة.
            InsufficientStockError: إذا تجاوزت الكمية المطلوبة المخزون المتاح.
        """
        if quantity <= 0:
            raise ValueError("الكمية يجب أن تكون موجبة")

        product = self.get_product(sku)

        if quantity > product.quantity:
            raise InsufficientStockError(
                f"الكمية المتوفرة ({product.quantity}) أقل من المطلوبة ({quantity})"
            )

        product.quantity -= quantity
        return product.price * quantity

    def total_inventory_value(self) -> float:
        """حساب القيمة الإجمالية لكل المخزون الحالي."""
        return sum(p.price * p.quantity for p in self._products.values())

    def low_stock_products(self, threshold: int = 5) -> list[Product]:
        """إرجاع قائمة المنتجات التي كميتها أقل من أو تساوي حدًا معيّنًا."""
        return [p for p in self._products.values() if p.quantity <= threshold]
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `tests/conftest.py` — Fixtures مشتركة بين كل ملفات الاختبار

```python
"""Fixtures مشتركة متاحة تلقائيًا لكل ملفات الاختبار في هذا المجلد."""

import pytest

from inventory.inventory_manager import InventoryManager
from inventory.models import Product

@pytest.fixture
def empty_manager():
    """مدير مخزون فارغ تمامًا."""
    return InventoryManager()

@pytest.fixture
def manager_with_products(empty_manager):
    """مدير مخزون مُعبّأ بمنتجات تجريبية جاهزة للاختبارات."""
    empty_manager.add_product(Product(sku="LAP001", name="لابتوب", price=3500, quantity=10))
    empty_manager.add_product(Product(sku="MOU001", name="ماوس", price=50, quantity=100))
    empty_manager.add_product(Product(sku="KEY001", name="لوحة مفاتيح", price=120, quantity=3))
    return empty_manager
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

> **💡 نصيحة خبير:** ملف `conftest.py` ملف خاص يكتشفه pytest تلقائيًا — أي fixture مُعرَّفة فيه تصبح متاحة **تلقائيًا** في كل ملفات الاختبار داخل نفس المجلد (وما دونه)، دون الحاجة لاستيرادها يدويًا. هذا هو المكان القياسي لوضع الـ fixtures المشتركة بين عدة ملفات اختبار.

### `tests/test_inventory_manager.py`

```python
"""اختبارات شاملة لنظام إدارة المخزون."""

import pytest

from inventory.models import (
    DuplicateSKUError,
    InsufficientStockError,
    Product,
    ProductNotFoundError,
)

class TestAddProduct:
    """اختبارات إضافة منتج جديد."""

    def test_add_product_succeeds(self, empty_manager):
        product = Product(sku="TEST01", name="منتج تجريبي", price=100, quantity=5)
        empty_manager.add_product(product)

        retrieved = empty_manager.get_product("TEST01")
        assert retrieved.name == "منتج تجريبي"

    def test_add_duplicate_sku_raises_error(self, manager_with_products):
        duplicate = Product(sku="LAP001", name="لابتوب آخر", price=4000, quantity=5)

        with pytest.raises(DuplicateSKUError):
            manager_with_products.add_product(duplicate)

class TestGetProduct:
    """اختبارات استرجاع منتج."""

    def test_get_existing_product(self, manager_with_products):
        product = manager_with_products.get_product("MOU001")
        assert product.name == "ماوس"
        assert product.price == 50

    def test_get_nonexistent_product_raises_error(self, manager_with_products):
        with pytest.raises(ProductNotFoundError):
            manager_with_products.get_product("NOTEXIST")

class TestSellProduct:
    """اختبارات عملية البيع."""

    def test_sell_reduces_quantity(self, manager_with_products):
        manager_with_products.sell("LAP001", 3)
        remaining = manager_with_products.get_product("LAP001")
        assert remaining.quantity == 7

    def test_sell_returns_correct_revenue(self, manager_with_products):
        revenue = manager_with_products.sell("MOU001", 10)
        assert revenue == 500  # 50 * 10

    def test_sell_nonexistent_product_raises_error(self, manager_with_products):
        with pytest.raises(ProductNotFoundError):
            manager_with_products.sell("NOTEXIST", 1)

    @pytest.mark.parametrize("invalid_quantity", [0, -1, -100])
    def test_sell_non_positive_quantity_raises_error(
        self, manager_with_products, invalid_quantity
    ):
        with pytest.raises(ValueError):
            manager_with_products.sell("LAP001", invalid_quantity)

    def test_sell_more_than_available_raises_error(self, manager_with_products):
        with pytest.raises(InsufficientStockError):
            manager_with_products.sell("KEY001", 1000)

    def test_sell_exact_remaining_quantity_succeeds(self, manager_with_products):
        """حالة حدّية: بيع الكمية المتبقية بالضبط يجب أن ينجح (وليس يرفض)."""
        manager_with_products.sell("KEY001", 3)  # الكمية المتوفرة بالضبط
        remaining = manager_with_products.get_product("KEY001")
        assert remaining.quantity == 0

class TestInventoryReports:
    """اختبارات تقارير المخزون الإجمالية."""

    def test_total_inventory_value(self, manager_with_products):
        # (3500*10) + (50*100) + (120*3) = 35000 + 5000 + 360 = 40360
        assert manager_with_products.total_inventory_value() == 40360

    def test_empty_inventory_value_is_zero(self, empty_manager):
        assert empty_manager.total_inventory_value() == 0

    def test_low_stock_products_default_threshold(self, manager_with_products):
        low_stock = manager_with_products.low_stock_products()
        skus = [p.sku for p in low_stock]
        assert "KEY001" in skus      # كمية 3، أقل من الحد الافتراضي 5
        assert "LAP001" not in skus  # كمية 10، أعلى من الحد

    def test_low_stock_products_custom_threshold(self, manager_with_products):
        low_stock = manager_with_products.low_stock_products(threshold=15)
        skus = [p.sku for p in low_stock]
        assert "LAP001" in skus  # كمية 10، أقل من الحد المخصص 15
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### تشغيل المجموعة الكاملة مع تقرير التغطية

```bash
uv run pytest --cov=src --cov-report=term-missing -v
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
tests/test_inventory_manager.py::TestAddProduct::test_add_product_succeeds PASSED
tests/test_inventory_manager.py::TestAddProduct::test_add_duplicate_sku_raises_error PASSED
tests/test_inventory_manager.py::TestGetProduct::test_get_existing_product PASSED
tests/test_inventory_manager.py::TestGetProduct::test_get_nonexistent_product_raises_error PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_reduces_quantity PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_returns_correct_revenue PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_nonexistent_product_raises_error PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_non_positive_quantity_raises_error[0] PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_non_positive_quantity_raises_error[-1] PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_non_positive_quantity_raises_error[-100] PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_more_than_available_raises_error PASSED
tests/test_inventory_manager.py::TestSellProduct::test_sell_exact_remaining_quantity_succeeds PASSED
tests/test_inventory_manager.py::TestInventoryReports::test_total_inventory_value PASSED
tests/test_inventory_manager.py::TestInventoryReports::test_empty_inventory_value_is_zero PASSED
tests/test_inventory_manager.py::TestInventoryReports::test_low_stock_products_default_threshold PASSED
tests/test_inventory_manager.py::TestInventoryReports::test_low_stock_products_custom_threshold PASSED

---------- coverage: platform linux, python 3.14.0 -----------
Name                                  Stmts   Miss  Cover
---------------------------------------------------------
src/inventory/models.py                 14      0   100%
src/inventory/inventory_manager.py      28      0   100%
---------------------------------------------------------
TOTAL                                   42      0   100%

========================= 16 passed in 0.05s =========================
```

تغطية 100%، و16 اختبارًا يُغطّون الحالات الطبيعية **والحدّية والاستثنائية** — بثقة كاملة أن هذا النظام يعمل كما هو متوقع.

<details>
<summary>✅ الحل الكامل للمشروع (جميع الملفات مجتمعة)</summary>

جميع ملفات `models.py`، `inventory_manager.py`، `conftest.py`، و`test_inventory_manager.py` كما وردت بالكامل أعلاه تُشكّل الحل النموذجي الكامل.

```bash
uv add --dev pytest pytest-cov
uv run pytest --cov=src --cov-report=term-missing -v
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**نقاط تصميم مهمة في هذا الحل:**
1. **`conftest.py` للـ Fixtures المشتركة**: يتجنّب تكرار إعداد بيانات الاختبار في كل ملف.
2. **تنظيم الاختبارات في أصناف** (`TestAddProduct`, `TestSellProduct`...) يعكس بنية الدوال المُختبَرة ويُسهّل التنقّل.
3. **تغطية شاملة للحالات**: الحالة الطبيعية، الحدّية (الكمية المتبقية بالضبط)، والاستثنائية (منتج غير موجود، كمية سالبة، مخزون غير كافٍ) لكل دالة.
4. **استخدام `parametrize`** لاختبار عدة قيم غير صالحة بكفاءة دون تكرار الكود.
5. **استثناءات مخصصة واضحة** (كما تعلمنا في الفصل 11) تجعل اختبار حالات الخطأ دقيقًا ومعبّرًا.

</details>

## 25.12 متى تكتب الاختبارات؟ (TDD بإيجاز)

> **💡 نصيحة خبير: التطوير الموجَّه بالاختبار (Test-Driven Development - TDD)**
> منهجية TDD تقترح كتابة **الاختبار أولاً**، قبل كتابة الكود الفعلي، بدورة من ثلاث خطوات:
> 1. 🔴 **Red**: اكتب اختبارًا يفشل (لأن الكود غير موجود بعد).
> 2. 🟢 **Green**: اكتب أبسط كود ممكن لجعل الاختبار ينجح.
> 3. 🔵 **Refactor**: حسّن الكود (تطبيق مبادئ الفصل 24) مع إبقاء الاختبار ناجحًا.
>
> هذه المنهجية ليست إلزامية للجميع، لكنها تُجبرك على التفكير بوضوح في «ماذا يُفترض أن يفعل الكود بالضبط» **قبل** الغرق في التفاصيل التقنية. جرّبها على مشاريعك الصغيرة القادمة لترى إن كانت تناسب أسلوبك.

> **⚠️ تحذير: لا تنتظر «حتى ينتهي المشروع» لكتابة الاختبارات**
> الاعتقاد الشائع الخاطئ: «سأكتب الاختبارات بعد أن أنهي كل الميزات.» في الواقع، هذا نادرًا ما يحدث — الضغط لتسليم ميزات جديدة يجعل الاختبارات تُؤجَّل إلى الأبد. الممارسة الأفضل: اكتب اختبارًا **لكل دالة جديدة مهمة فور الانتهاء منها**، كجزء طبيعي من عملية التطوير، وليس كخطوة منفصلة «لاحقة».

## 25.13 تمارين

### 🟢 مستوى مبتدئ

1. اكتب دالة `is_palindrome(text: str) -> bool` تتحقق إن كانت الكلمة تُقرأ نفسها من الاتجاهين، ثم اكتب 4 اختبارات لها (حالة صحيحة، حالة خاطئة، نص فارغ، نص بحرف واحد).
2. استخدم `pytest.raises` لاختبار أن دالة `calculate_average(numbers: list)` ترفع `ValueError` عند استقبال قائمة فارغة.
3. استخدم `@pytest.mark.parametrize` لاختبار دالة `is_even(n: int) -> bool` بـ 5 قيم مختلفة (زوجية وفردية، موجبة وسالبة، صفر).

<details>
<summary>✅ الحل المقترح</summary>

```python
# دوال للاختبار (ضعها في ملف منفصل src/utils.py)
def is_palindrome(text: str) -> bool:
    cleaned = text.lower().replace(" ", "")
    return cleaned == cleaned[::-1]

def calculate_average(numbers: list) -> float:
    if not numbers:
        raise ValueError("القائمة لا يمكن أن تكون فارغة")
    return sum(numbers) / len(numbers)

def is_even(n: int) -> bool:
    return n % 2 == 0

# الاختبارات
import pytest
from utils import is_palindrome, calculate_average, is_even

def test_is_palindrome_true_case():
    assert is_palindrome("مدم") is True

def test_is_palindrome_false_case():
    assert is_palindrome("بيت") is False

def test_is_palindrome_empty_string():
    assert is_palindrome("") is True

def test_is_palindrome_single_char():
    assert is_palindrome("أ") is True

def test_calculate_average_empty_list_raises_error():
    with pytest.raises(ValueError):
        calculate_average([])

@pytest.mark.parametrize("n, expected", [
    (2, True), (3, False), (0, True), (-4, True), (-7, False),
])
def test_is_even(n, expected):
    assert is_even(n) == expected
```

🧾 **للقراءة فقط** — الوحدة `utils` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

</details>

### 🟡 مستوى متوسط

4. أضف fixture `sample_products` تُرجع قائمة من 3 منتجات تجريبية، ثم اكتب اختبارًا لدالة تحسب متوسط أسعارها.
5. اكتب اختبارًا يستخدم `unittest.mock.patch` لمحاكاة دالة `datetime.now()` بحيث تُرجع تاريخًا ثابتًا، لاختبار دالة تعتمد على التاريخ الحالي بشكل متسق (بدون اعتماد على التاريخ الفعلي وقت تشغيل الاختبار).

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
# 4
import pytest

@pytest.fixture
def sample_products():
    return [
        {"name": "أ", "price": 100},
        {"name": "ب", "price": 200},
        {"name": "ج", "price": 300},
    ]

def average_price(products: list[dict]) -> float:
    return sum(p["price"] for p in products) / len(products)

def test_average_price(sample_products):
    assert average_price(sample_products) == 200

# 5
from datetime import datetime
from unittest.mock import patch

def get_current_year() -> int:
    return datetime.now().year

def test_get_current_year_with_fixed_date():
    fixed_date = datetime(2030, 1, 1)
    with patch("__main__.datetime") as mock_datetime:
        mock_datetime.now.return_value = fixed_date
        assert get_current_year() == 2030

</details>

### 🔴 مستوى تحدي

6. وسّع مشروع إدارة المخزون: أضف دالة جديدة `restock(sku: str, quantity: int)` تزيد كمية منتج موجود، مع رفع `ValueError` إذا كانت الكمية المُضافة غير موجبة. اكتب مجموعة اختبارات كاملة لها (الحالة الطبيعية، الكمية الصفرية/السالبة، منتج غير موجود)، واحرص على تحقيق تغطية 100% لهذه الدالة الجديدة تحديدًا.

<details>
<summary>✅ الحل المقترح</summary>

```python
# إضافة إلى inventory_manager.py
def restock(self, sku: str, quantity: int) -> None:
    """زيادة كمية منتج موجود في المخزون.

    Raises:
        ProductNotFoundError: إذا لم يوجد المنتج.
        ValueError: إذا كانت الكمية المُضافة غير موجبة.
    """
    if quantity <= 0:
        raise ValueError("كمية التزويد يجب أن تكون موجبة")

    product = self.get_product(sku)  # يرفع ProductNotFoundError تلقائيًا إن لم يوجد
    product.quantity += quantity

# الاختبارات
class TestRestock:
    def test_restock_increases_quantity(self, manager_with_products):
        manager_with_products.restock("KEY001", 10)
        product = manager_with_products.get_product("KEY001")
        assert product.quantity == 13

    @pytest.mark.parametrize("invalid_quantity", [0, -1, -50])
    def test_restock_non_positive_raises_error(self, manager_with_products, invalid_quantity):
        with pytest.raises(ValueError):
            manager_with_products.restock("KEY001", invalid_quantity)

    def test_restock_nonexistent_product_raises_error(self, manager_with_products):
        with pytest.raises(ProductNotFoundError):
            manager_with_products.restock("NOTEXIST", 10)
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `ProductNotFoundError` من مثال سابق.

</details>

## 25.14 اختبر فهمك (Quiz)

<details>
<summary>1. ما معنى نمط Arrange-Act-Assert وما فائدته؟</summary>

هو نمط تنظيمي لكتابة الاختبارات بثلاث خطوات واضحة: Arrange (تحضير البيانات والظروف اللازمة)، Act (تنفيذ الكود المراد اختباره)، Assert (التحقق من أن النتيجة مطابقة للمتوقع). فائدته أنه يجعل كل اختبار واضح البنية وسهل القراءة والفهم، ويفصل بوضوح بين «ماذا نحضّر» و«ماذا نختبر فعليًا».
</details>

<details>
<summary>2. ما وظيفة الـ Fixtures في pytest ولماذا هي مفيدة؟</summary>

الـ Fixtures دوال خاصة تُحضّر بيانات أو موارد (كاتصال وهمي، كائن تجريبي، ملف مؤقت) وتُمرّرها تلقائيًا لدوال الاختبار التي تحتاجها، عبر تمريرها كمعامل بنفس الاسم. فائدتها الأساسية هي تجنّب تكرار كود «التحضير» في كل اختبار على حدة، وتوحيد طريقة إعداد البيانات المشتركة بين عدة اختبارات.
</details>

<details>
<summary>3. متى يجب استخدام Mocking بدلاً من استدعاء الكود الحقيقي مباشرة في الاختبار؟</summary>

عند الاعتماد على موارد خارجية بطيئة، غير موثوقة، أو مكلفة، مثل طلبات الشبكة الحقيقية، قواعد البيانات الفعلية، أو خدمات خارجية (كإرسال بريد إلكتروني). استخدام Mocking يجعل الاختبارات سريعة وموثوقة ومستقلة تمامًا عن توفر هذه الموارد الخارجية وقت تشغيل الاختبار.
</details>

<details>
<summary>4. لماذا تغطية الاختبار 100% ليست دليلًا كافيًا على أن الكود خالٍ من الأخطاء؟</summary>

لأن التغطية تقيس فقط أن كل سطر من الكود تم **تنفيذه** مرة واحدة على الأقل أثناء الاختبارات، لكنها لا تضمن أن كل **سيناريو منطقي ممكن** (بما فيه الحالات الحدّية والاستثنائية) قد اختُبر بعمق. يمكن تحقيق تغطية 100% باختبارات سطحية جدًا تتجاهل حالات مهمة.
</details>

<details>
<summary>5. ما الفرق بين استخدام `@pytest.mark.parametrize` وكتابة عدة دوال اختبار منفصلة لنفس المنطق؟</summary>

`parametrize` يسمح بتشغيل نفس منطق الاختبار بعدة مجموعات من المدخلات والنتائج المتوقعة من خلال دالة اختبار واحدة فقط، مما يقلل تكرار الكود (تطبيق مبدأ DRY) ويجعل إضافة حالة اختبار جديدة بسيطة (مجرد سطر جديد في القائمة)، بدلاً من نسخ ولصق دالة اختبار كاملة لكل حالة جديدة.
</details>

## 25.15 ملخص الفصل (Cheat Sheet)

```python
# ===== الأساسيات =====
def test_something():                    # يجب أن يبدأ بـ test_
    assert actual == expected

# ===== اختبار الاستثناءات =====
import pytest
with pytest.raises(ValueError):
    risky_function()
with pytest.raises(ValueError, match="رسالة محددة"):
    risky_function()

# ===== الأعداد العشرية =====
assert 0.1 + 0.2 == pytest.approx(0.3)

# ===== Fixtures =====
@pytest.fixture
def my_data():
    return {"key": "value"}

def test_uses_fixture(my_data):
    assert my_data["key"] == "value"

@pytest.fixture
def resource_with_cleanup():
    resource = setup()
    yield resource
    teardown(resource)

# ===== Parametrize =====
@pytest.mark.parametrize("a, b, expected", [(1,1,2), (2,2,4)])
def test_add(a, b, expected):
    assert add(a, b) == expected

# ===== Mocking =====
from unittest.mock import patch, MagicMock
with patch("module.function_name") as mock_func:
    mock_func.return_value = "قيمة وهمية"

# ===== أوامر التشغيل =====
# uv run pytest                              تشغيل كل الاختبارات
# uv run pytest -v                           تفصيلي
# uv run pytest tests/test_x.py::test_y      اختبار محدد
# uv run pytest --cov=src                    مع تقرير التغطية
# uv run pytest -k "keyword"                 فلترة بالاسم
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `actual`، `risky_function`، `setup` من مثال سابق.

| الملف/المفهوم | الدور |
|---|---|
| `test_*.py` | ملفات الاختبار (تُكتشف تلقائيًا) |
| `conftest.py` | fixtures مشتركة بين ملفات الاختبار |
| `pytest.fixture` | تحضير بيانات/موارد للاختبارات |
| `pytest.raises` | اختبار أن كودًا يرفع استثناءً متوقعًا |
| `pytest.mark.parametrize` | تشغيل نفس الاختبار بمدخلات متعددة |
| `unittest.mock.patch` | محاكاة اعتماديات خارجية |

## 25.16 ماذا بعد؟

لقد تعلّمت الآن المهارة التي تُميّز المبرمج المحترف عن الهاوي بشكل حاسم: **الثقة في كودك**، مدعومة باختبارات آلية تتحقق من صحته باستمرار. مع كل ما تعلمته من كود نظيف (الفصل 24) واختبارات شاملة (هذا الفصل)، أصبح لديك الآن الأساس الكامل لكتابة برمجيات احترافية جاهزة للعالم الحقيقي.

في الفصل الأخير من رحلتنا، سنُكمل الصورة: كيف تُحوّل مشروعك من مجرد «كود يعمل على جهازك» إلى **منتج حقيقي** يمكن للآخرين تثبيته واستخدامه؟ سنتعلّم أساسيات تحزيم المشاريع (Packaging)، نظرة سريعة على مفهوم النشر (Deployment)، ثم نختم بخارطة طريق شاملة لما بعد هذا الكتاب — إلى أين تذهب من هنا في رحلتك كمبرمج بايثون.

---

### الخطوة التالية

- [الفصل السادس والعشرون: من الكود إلى المنتج (Packaging, Deployment، وخارطة الطريق)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-26.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الخامس والعشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*